In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
df_returns_raw = spark.read.parquet("abfss://e8b72d8d-5a0f-40b4-bcfc-e5ff1552d786@onelake.dfs.fabric.microsoft.com/ac40a136-0e9a-48e0-ab5d-652add7f1fed/Files/bronze/returns_data")
display(df_returns_raw)

In [ ]:
#Extract First Row
first_row=df_returns_raw.first()
columns=[str(item).strip() for item in first_row]

#Remove the first row (header row now part of data)
# 1. Convert DataFrame to RDD to use zipWithIndex
# 2. Filter out the first row (index 0)
# 3. Extract just the Row object from the tuple, ignoring the index
rdd_filtered = df_returns_raw.rdd.zipWithIndex() \
    .filter(lambda x: x[1] > 0) \
    .map(lambda x: x[0])

# 4. Reconstruct the DataFrame at the end using your column list
df_returns_raw = spark.createDataFrame(rdd_filtered, schema=columns)

# show cleaned data
display(df_returns_raw)

1. Rename the Required Col's

In [ ]:
df_returns_raw=df_returns_raw.withColumnRenamed("Return_ID", "ReturnID") \
    .withColumnRenamed("Order_ID", "OrderID") \
    .withColumnRenamed("Customer_ID", "CustomerID") \
    .withColumnRenamed("Return_Reason", "ReturnReason") \
    .withColumnRenamed("Return_Date", "ReturnDate") \
    .withColumnRenamed("Refund_Status", "RefundStatus") \
    .withColumnRenamed("Pickup_Address", "PickupAddress") \
    .withColumnRenamed("Return_Amount", "ReturnAmount") 

display(df_returns_raw)

In [ ]:
from pyspark.sql.functions import *
df_returns=df_returns_raw.withColumn("RefundStatus", lower(regexp_replace(col("RefundStatus"), r"[^a-zA-Z]", "")))
display(df_returns)

In [ ]:
from pyspark.sql.types import DoubleType
df_returns=df_returns.withColumn("ReturnAmount", 
        regexp_extract(col("ReturnAmount"), r"(\d+\.?\d*)", 1).cast(DoubleType())
    )

display(df_returns)

In [ ]:
from pyspark.sql.functions import col, coalesce, to_date

# Overwrite the existing column with the standardized date type
df_returns = df_returns.withColumn(
    "ReturnDate",
    coalesce(
        to_date(col("ReturnDate"), "yyyy-MM-dd"),
        to_date(col("ReturnDate"), "yyyy/MM/dd"),
        to_date(col("ReturnDate"), "yyyy.MM.dd"),
        to_date(col("ReturnDate"), "dd-MM-yyyy"),
        to_date(col("ReturnDate"), "dd/MM/yyyy"),
        to_date(col("ReturnDate"), "dd.MM.yyyy"),
        to_date(col("ReturnDate"), "MM/dd/yyyy")
    )
)

# Verify the changes and the updated schema
df_returns.show(15)
#display(df_returns)

In [ ]:
display(df_returns)

In [ ]:
# 2.5 Clean PickupAddress → remove special characters
df_returns=df_returns.withColumn("PickupAddress", initcap(trim(regexp_replace(col("PickupAddress"), r"[^a-zA-Z0-9\s]", " "))))
    
# 2.6 Clean Product → remove extra symbols and spaces
df_returns=df_returns.withColumn("Product", initcap(trim(regexp_replace(col("Product"), r"[^a-zA-Z0-9\s]", ""))))
    
# 2.7 Clean CustomerID → trim, fix wrong prefixes
df_returns=df_returns.withColumn("CustomerID", trim(upper(col("CustomerID"))))

display(df_returns)

In [ ]:
df_returns=df_returns.filter(col("ReturnID").isNotNull())
display(df_returns)

In [ ]:
df_returns.write.mode("overwrite").format("delta").saveAsTable("silver_returns")